# WDR91 DEL — Model Training & Evaluation

**Goal:** Train and compare XGBoost and LightGBM classifiers on combined fingerprints (ECFP6 + MACCS + RDK) from the DEL screen.

**Primary metric:** Enrichment Factor @1% (EF@1%)

**Split strategy:** Stratified 80/20 train/test split.

**Features:** ECFP6 (2048) + MACCS (167) + RDK (2048) = **4263 bits** per compound

---
**Before running:** Upload `WDR91.parquet` to `My Drive/CS502/data/WDR91.parquet`

In [ ]:
!pip install xgboost lightgbm pyarrow tqdm seaborn -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH      = '/content/drive/MyDrive/CS502/data/WDR91.parquet'
XGB_MODEL_PATH = '/content/drive/MyDrive/CS502/models/xgb_multifp.json'
LGB_MODEL_PATH = '/content/drive/MyDrive/CS502/models/lgbm_multifp.txt'

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow.parquet as pq
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    balanced_accuracy_score, recall_score,
    RocCurveDisplay, PrecisionRecallDisplay
)
from pathlib import Path
from tqdm import tqdm

%matplotlib inline
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

Path('/content/drive/MyDrive/CS502/models').mkdir(parents=True, exist_ok=True)

## Helper Functions

In [ ]:
FP_DIMS = {
    'ECFP4': 2048, 'ECFP6': 2048, 'FCFP4': 2048, 'FCFP6': 2048,
    'MACCS': 167, 'RDK': 2048, 'AVALON': 512, 'ATOMPAIR': 2048, 'TOPTOR': 2048,
}

SCALAR_COLS = [
    'COMPOUND_ID', 'LIBRARY_ID', 'BB1_ID', 'BB2_ID', 'BB3_ID',
    'TARGET_VALUE', 'LABEL', 'MW', 'ALOGP'
]

FP_NAMES = ['ECFP6', 'MACCS', 'RDK']


def indices_to_sparse(index_lists, n_bits):
    rows, cols = [], []
    for i, bits in enumerate(index_lists):
        for b in bits:
            if b < n_bits:
                rows.append(i)
                cols.append(b)
    data = np.ones(len(rows), dtype=np.uint8)
    return sp.csr_matrix((data, (rows, cols)), shape=(len(index_lists), n_bits))


def load_multi_fingerprints(path, fp_names=FP_NAMES, batch_size=50_000):
    pf = pq.ParquetFile(path)
    scalar_frames = []
    fp_batches = {fp: [] for fp in fp_names}

    for batch in tqdm(pf.iter_batches(batch_size=batch_size, columns=SCALAR_COLS + fp_names),
                      desc='Loading fingerprints'):
        batch_df = batch.to_pandas()
        scalar_frames.append(batch_df[SCALAR_COLS])
        for fp in fp_names:
            fp_batches[fp].append(indices_to_sparse(batch_df[fp].tolist(), FP_DIMS[fp]))

    fp_matrices = [sp.vstack(fp_batches[fp], format='csr') for fp in fp_names]
    X = sp.hstack(fp_matrices, format='csr')
    df = pd.concat(scalar_frames, ignore_index=True)
    total_bits = sum(FP_DIMS[fp] for fp in fp_names)
    print(f'Combined feature matrix: {X.shape}  ({total_bits} bits from {fp_names})')
    return X, df


def enrichment_factor(y_true, y_score, frac=0.01):
    n = len(y_true)
    n_top = max(1, int(n * frac))
    top_idx = np.argsort(y_score)[::-1][:n_top]
    hits_in_top = y_true[top_idx].sum()
    total_hits = y_true.sum()
    if total_hits == 0:
        return 0.0
    return (hits_in_top / n_top) / (total_hits / n)


def evaluate(y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    return {
        'ROC-AUC':   roc_auc_score(y_true, y_prob),
        'PR-AUC':    average_precision_score(y_true, y_prob),
        'Bal. Acc.': balanced_accuracy_score(y_true, y_pred),
        'Recall':    recall_score(y_true, y_pred),
        'EF@1%':     enrichment_factor(y_true, y_prob, 0.01),
        'EF@5%':     enrichment_factor(y_true, y_prob, 0.05),
    }

## 1. Load Data

In [ ]:
print(f'Loading fingerprints: {FP_NAMES} ...')
X, df = load_multi_fingerprints(DATA_PATH)

y = df['LABEL'].values

print(f'\nX shape : {X.shape}')
print(f'Hits    : {y.sum():,} / {len(y):,} ({100*y.mean():.2f}%)')

## 2. Stratified 80/20 Train/Test Split

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_tr_f32 = X_tr.astype(np.float32)
X_te_f32 = X_te.astype(np.float32)

SPW = (y_tr == 0).sum() / (y_tr == 1).sum()
print(f'Train : {X_tr.shape[0]:,} compounds  ({y_tr.sum():,} hits, {100*y_tr.mean():.2f}%)')
print(f'Test  : {X_te.shape[0]:,} compounds  ({y_te.sum():,} hits, {100*y_te.mean():.2f}%)')
print(f'scale_pos_weight = {SPW:.2f}')

## 3. Train XGBoost

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=SPW,
    eval_metric='aucpr',
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)
xgb_model.fit(X_tr_f32, y_tr)
print('XGBoost training complete.')

## 4. Train LightGBM

In [ ]:
lgb_model = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=SPW,
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)
lgb_model.fit(X_tr_f32, y_tr)
print('LightGBM training complete.')

## 5. Compare Results

In [ ]:
xgb_prob = xgb_model.predict_proba(X_te_f32)[:, 1]
lgb_prob = lgb_model.predict_proba(X_te_f32)[:, 1]

fp_label = '+'.join(FP_NAMES)
results_df = pd.DataFrame([
    {'Features': fp_label, 'Model': 'XGBoost',  **evaluate(y_te, xgb_prob)},
    {'Features': fp_label, 'Model': 'LightGBM', **evaluate(y_te, lgb_prob)},
])

print('\n=== Test Set Results ===')
print(results_df.to_string(index=False))

## 6. ROC & Precision-Recall Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for prob, name, color in [
    (xgb_prob, 'XGBoost',  'coral'),
    (lgb_prob, 'LightGBM', 'steelblue'),
]:
    RocCurveDisplay.from_predictions(y_te, prob, ax=axes[0],
        name=f'{name} (AUC={roc_auc_score(y_te, prob):.3f})')
    PrecisionRecallDisplay.from_predictions(y_te, prob, ax=axes[1],
        name=f'{name} (AP={average_precision_score(y_te, prob):.3f})')

axes[0].plot([0,1],[0,1],'k--', alpha=0.4)
axes[0].set_title('ROC Curve (Test Set)')
axes[1].axhline(y_te.mean(), color='k', linestyle='--', alpha=0.4,
                label=f'Random ({y_te.mean():.3f})')
axes[1].legend(fontsize=8)
axes[1].set_title('Precision-Recall Curve (Test Set)')

plt.tight_layout()
plt.show()

## 7. Enrichment Factor Curves

In [ ]:
fractions = np.linspace(0.001, 0.20, 100)

fig, ax = plt.subplots(figsize=(9, 5))
for prob, name, color in [
    (xgb_prob, 'XGBoost',  'coral'),
    (lgb_prob, 'LightGBM', 'steelblue'),
]:
    efs = [enrichment_factor(y_te, prob, f) for f in fractions]
    ax.plot(fractions * 100, efs, lw=2, color=color, label=name)

ax.axhline(1.0, color='gray', linestyle='--', alpha=0.6, label='Random baseline')
ax.axvline(1.0, color='black', linestyle=':', alpha=0.5, label='1% cutoff')
ax.set_xlabel('Top X% of ranked list')
ax.set_ylabel('Enrichment Factor')
ax.set_title('Enrichment Factor Curve (Test Set)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'\n{"":20s} {"XGBoost":>10s} {"LightGBM":>10s}')
for frac, label in [(0.001,'0.1%'),(0.005,'0.5%'),(0.01,'1%'),(0.05,'5%')]:
    ef_xgb = enrichment_factor(y_te, xgb_prob, frac)
    ef_lgb = enrichment_factor(y_te, lgb_prob, frac)
    print(f'EF @{label:4s}           {ef_xgb:>10.2f} {ef_lgb:>10.2f}')

## 8. Train Final Models on Full Data & Save

In [ ]:
SPW_full = (y == 0).sum() / (y == 1).sum()
X_full = X.astype(np.float32)

print('Training final XGBoost on full dataset...')
final_xgb = xgb.XGBClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, scale_pos_weight=SPW_full,
    eval_metric='aucpr', random_state=42, n_jobs=-1, verbosity=0,
)
final_xgb.fit(X_full, y)
final_xgb.save_model(XGB_MODEL_PATH)
print(f'XGBoost saved to {XGB_MODEL_PATH}')

print('\nTraining final LightGBM on full dataset...')
final_lgb = lgb.LGBMClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, scale_pos_weight=SPW_full,
    random_state=42, n_jobs=-1, verbosity=-1,
)
final_lgb.fit(X_full, y)
final_lgb.booster_.save_model(LGB_MODEL_PATH)
print(f'LightGBM saved to {LGB_MODEL_PATH}')

## 9. Feature Importance by Fingerprint Block (XGBoost)

In [ ]:
importance = final_xgb.feature_importances_
boundaries = np.cumsum([0] + [FP_DIMS[fp] for fp in FP_NAMES])

fp_importance = {}
for i, fp in enumerate(FP_NAMES):
    fp_importance[fp] = importance[boundaries[i]:boundaries[i+1]].sum()

total_imp = sum(fp_importance.values())
print('Feature importance by fingerprint block (XGBoost):')
for fp, imp in fp_importance.items():
    print(f'  {fp:8s}: {imp:.4f}  ({100*imp/total_imp:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(fp_importance.keys(), fp_importance.values(),
            color=['coral', 'steelblue', 'mediumpurple'])
axes[0].set_ylabel('Total importance (gain)')
axes[0].set_title('Importance by Fingerprint Type (XGBoost)')

top_n = 30
top_idx = np.argsort(importance)[::-1][:top_n]
bit_labels = []
for idx in top_idx:
    for i, fp in enumerate(FP_NAMES):
        if boundaries[i] <= idx < boundaries[i+1]:
            bit_labels.append(f'{fp}:{idx - boundaries[i]}')
            break

axes[1].bar(range(top_n), importance[top_idx], color='mediumpurple')
axes[1].set_xticks(range(top_n))
axes[1].set_xticklabels(bit_labels, rotation=90, fontsize=7)
axes[1].set_ylabel('Importance (gain)')
axes[1].set_title(f'Top {top_n} Most Important Bits (XGBoost)')

plt.tight_layout()
plt.show()